# Block 9 — LAB: Handling Outliers — Detecting & Treating Them
### Advanced Machine Learning — M&T Bank

Loads `bank_marketing_features.csv` (unchanged since Block 3).

**Part 1 — Detect:** run the IQR method and the Z-score method on `campaign` and `duration`, and compare what each
one flags.

**Part 2 — Treat:** cap (winsorize) and log-transform `campaign`, then check whether it actually changes what our
model learns.

**Part 3 — The trap:** run the same generic method on `pdays` and see what happens when a column isn't really
continuous.

**Output:** `bank_marketing_outliers.csv` — the same 41,188 rows and 50 columns as `bank_marketing_features.csv`,
plus four new columns. `bank_marketing_features.csv` itself is not touched.

Look for `# TODO` — that's where your code goes. Each task has a hint; ask if you get stuck.


## Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.max_columns", 30)

NAVY = "#251E4E"
PINK = "#FF1675"
GRAY = "#6b7280"
ORANGE = "#FF7B01"

# Data source: https://raw.githubusercontent.com/mithun-rk/mt-advml-course/main/Data/bank_marketing_features.csv
df = pd.read_csv("https://raw.githubusercontent.com/mithun-rk/mt-advml-course/main/Data/bank_marketing_features.csv", sep=";")
print(df.shape)
df[["campaign", "duration", "pdays"]].describe()


## Part 1 — Detecting Outliers: IQR vs. Z-Score

**IQR method:** flag anything outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`

**Z-score method:** flag anything with `|z| > 3`

### TODO 1.1 — Write a helper that returns the IQR bounds for a column

Hint: `s.quantile(0.25)`, `s.quantile(0.75)`, then `Q1 - 1.5*IQR` and `Q3 + 1.5*IQR`.


In [ ]:
def iqr_bounds(s):
    # TODO: compute Q1, Q3, IQR and return (lower, upper)
    pass


### TODO 1.2 — For `campaign` and `duration`, compute both outlier counts

For each column: get the IQR bounds, build a boolean mask for IQR outliers, compute a Z-score column and build a
boolean mask for `|z| > 3`. Print (or collect into a small DataFrame) the count and percentage each method flags,
plus each column's skew (`s.skew()`).


In [ ]:
results = []
for col in ["campaign", "duration"]:
    s = df[col]
    # TODO: lower, upper = iqr_bounds(s)
    # TODO: iqr_mask = ...
    # TODO: z = (s - s.mean()) / s.std()
    # TODO: z_mask = ...
    # TODO: append a dict with column, skew, bounds, iqr count/pct, z count/pct to results
    pass

pd.DataFrame(results)


**Question to answer before moving on:** which method flags more points, and does that match what you'd expect
given each column's skew? (A symmetric normal distribution has skew ≈ 0 — check how far `campaign` and `duration`
are from that.)


### TODO 1.3 — Plot it

For each of `campaign` and `duration`: a histogram with a vertical line at the IQR upper bound and a second
vertical line at the Z-score upper bound (`mean + 3*std`). Do the two lines land in noticeably different places?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col in zip(axes, ["campaign", "duration"]):
    s = df[col]
    # TODO: lower, upper = iqr_bounds(s)
    # TODO: z_upper = s.mean() + 3 * s.std()
    ax.hist(s, bins=60, color=NAVY, alpha=0.75)
    # TODO: ax.axvline(...) for both bounds, with labels
    ax.set_title(col, color=NAVY, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


## Part 2 — Treating Outliers: Capping vs. Log-Transform

### TODO 2.1 — Create `campaign_capped` and `campaign_log`

- `campaign_capped`: clip `campaign` to its IQR bounds (`.clip(lower=..., upper=...)`) — use `max(lower, 1)` since
  `campaign` can't sensibly go below 1 call
- `campaign_log`: `np.log1p(df["campaign"])`

Do the same for `duration` → `duration_capped`, `duration_log` (use `max(lower, 0)` for the capped lower bound).


In [ ]:
# TODO: compute IQR bounds for campaign, then:
# df["campaign_capped"] = ...
# df["campaign_log"] = ...

# TODO: same for duration -> duration_capped, duration_log

df[["campaign", "campaign_capped", "campaign_log"]].describe()


### TODO 2.2 — Does it change the model?

Refit the standard `LogisticRegression(max_iter=2000, solver="lbfgs", C=0.5)` from Blocks 4-8 three times, swapping
only the `campaign` column each time: raw, capped, log. Same feature prep and split as every prior block
(`test_size=0.2, random_state=42, stratify=y`; drop `education`, `y`, `duration`, and the new `_capped`/`_log`
columns from the feature list; scale the usual numeric columns). Compare test-set ROC-AUC across the three.

Hint: write one function that takes the campaign column name to use and returns the AUC, so you're not repeating
the fit/scale/split logic three times.


In [ ]:
feature_cols = [c for c in df.columns if c not in ("education", "y", "duration",
                                                     "campaign_capped", "campaign_log",
                                                     "duration_capped", "duration_log")]
bool_cols = [c for c in df[feature_cols].columns if df[c].dtype == bool]
y = df["y"].values

def fit_and_score(campaign_col):
    # TODO: build Xd from feature_cols, overwrite the "campaign" column with df[campaign_col]
    # TODO: cast bool_cols to int
    # TODO: train_test_split, scale numeric_to_scale (same list as Block 8), fit LogisticRegression(C=0.5)
    # TODO: return roc_auc_score(yte, proba)
    pass

# TODO: call fit_and_score for "campaign", "campaign_capped", "campaign_log" and compare


**Question to answer:** did treating `campaign` change the AUC meaningfully? If not, is that a failed experiment,
or a legitimate finding — and what would make outlier treatment matter more for a different model or metric?


## Part 3 — The Sentinel-Value Trap: Running the Same Method on `pdays`

`pdays` = "days since the client was last contacted in a previous campaign."

### TODO 3.1 — Check what fraction of `pdays` equals 999

Recall from Block 3: `999` is a sentinel meaning "never previously contacted," and Block 3 already engineered
`was_contacted_before` as a separate boolean feature for exactly this reason.


In [ ]:
# TODO: print the count and percentage of rows where pdays == 999


### TODO 3.2 — Run IQR and Z-score on `pdays` anyway

Use the same `iqr_bounds` helper and the same Z-score approach from Part 1. What are the bounds? How many rows get
flagged by each method — and do the two methods agree on *which* rows this time (compare the two boolean masks
directly, not just the counts)?


In [ ]:
# TODO: lower, upper = iqr_bounds(df["pdays"]); print the bounds
# TODO: compute iqr_flagged and z_flagged boolean masks
# TODO: print counts for each, and whether (iqr_flagged == z_flagged).all()


**Question to answer:** given what you found in 3.1, what do the IQR bounds on `pdays` actually end up being, and
why does that make every non-999 row get flagged? Is this a useful outlier finding, or something else wearing an
outlier finding's clothes?


## Saving the Output

### TODO 4.1 — Save the result

Save `df` (which now has the 4 new `_capped`/`_log` columns from Part 2 added to the original 50) to
`bank_marketing_outliers.csv`, same `sep=";"` convention as every prior CSV in this course. Don't overwrite
`bank_marketing_features.csv`.


In [ ]:
# TODO: df.to_csv("bank_marketing_outliers.csv", sep=";", index=False)
# TODO: print df.shape to confirm 41188 rows and 54 columns


## Wrap-Up

Write 2-3 sentences: what did IQR and Z-score disagree about, did outlier treatment change the model, and what did
the `pdays` check teach you about running generic statistical methods on a column without checking what it actually
represents first?
